# FactOwl

## 0. Setting up imports and environment variables

In [1]:
import argparse
import json
import logging
import numpy as np
import os
import pandas as pd

from huggingface_hub import login
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

from factowl import FactScorerSpedUpVLLM as FactScorer
from factowl.io import save_predictions, load_simple_json

/disk/4tb/sushko/conda/envs/ltf/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 07-04 19:47:18 [__init__.py:244] Automatically detected platform cuda.


2025-07-04 19:47:20,719	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
[nltk_data] Downloading package punkt to /home/sushko/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2"
os.environ["VLLM_LOG_LEVEL"] = "WARNING"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 1. Setting up the parameters for generation

In here, `hf_token` is needed to load the model from HuggingFace, `model_name` is the model's name. Point `data_dir` to the directory, where the downloaded wikipedia dump is located. Your data, which will be evaluated, should be placed in `file_path`. Additionally, create a cache directory and point towards it using `cache_dir` variable.

In [3]:
hf_token = 'hf_token'
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
cache_dir = './cachedir/'
data_dir = './wikipedia_dumps/'
file_path = './data/'
cnp = 5 # Number of context pages retrieved from Wikipedia API.
nsp = 10 # Number of relevant passages retrieved to support a single atomic fact.
context_type = 'wikipedia_api'
is_bio = False

## 2. Initialize VLLM engine for generation

We use VLLM to increase the efficiency of our factchecking engine.

In [4]:
vllm_model = LLM(
    model=model_name,
)

INFO 07-04 19:47:29 [config.py:823] This model supports multiple tasks: {'generate', 'embed', 'classify', 'score', 'reward'}. Defaulting to 'generate'.
INFO 07-04 19:47:30 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-04 19:47:31 [core.py:455] Waiting for init message from front-end.
INFO 07-04 19:47:31 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:01<00:04,  1.34s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.41s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:03<00:00,  1.03it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:04<00:00,  1.11s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:04<00:00,  1.15s/it]



INFO 07-04 19:47:39 [default_loader.py:272] Loading weights took 4.73 seconds
INFO 07-04 19:47:39 [gpu_model_runner.py:1624] Model loading took 14.9596 GiB and 5.795187 seconds
INFO 07-04 19:47:47 [backends.py:462] Using cache directory: /home/sushko/.cache/vllm/torch_compile_cache/62403aeb1d/rank_0_0 for vLLM's torch.compile
INFO 07-04 19:47:47 [backends.py:472] Dynamo bytecode transform time: 7.77 s
INFO 07-04 19:47:53 [backends.py:135] Directly load the compiled graph(s) for shape None from the cache, took 5.055 s
INFO 07-04 19:47:54 [monitor.py:34] torch.compile takes 7.77 s in total
INFO 07-04 19:47:55 [gpu_worker.py:227] Available KV cache memory: 55.10 GiB
INFO 07-04 19:47:56 [kv_cache_utils.py:715] GPU KV cache size: 451,408 tokens
INFO 07-04 19:47:56 [kv_cache_utils.py:719] Maximum concurrency for 8,192 tokens per request: 55.10x
INFO 07-04 19:48:21 [gpu_model_runner.py:2048] Graph capturing finished in 25 secs, took 0.52 GiB
INFO 07-04 19:48:21 [core.py:171] init engine (prof

## 3. Set up the names of the target files and evaluation setup.

In here, you can set up the names of the files to check and determine, whether you want to use npm for checking or not.

In [5]:
gen_name2abstain = {
    "ChatGPT": "generic",
    "InstructGPT": "generic",
}

gen_name2setup = {
    "ChatGPT": "retrieval+llama",
    "InstructGPT": "retrieval+llama",
}

gen_names = [
    "ChatGPT",
    "InstructGPT",
]

## 4. Start evaluation!

In [8]:
eval_dict = {}

for j, gn in enumerate(gen_names):
    st = gen_name2setup[gn]
    print(f"Evaluating {gn}")
    bi = "_BIO" if is_bio else ""

    json_p = os.path.join(file_path, f"{gn}.jsonl")
    abstain_type = gen_name2abstain[gn]
    atomic_facts_cache_dir = f"./cache/{gn}-{st}{bi}-{context_type}-p{cnp}-c{nsp}/"
    print(f"{atomic_facts_cache_dir=}")

    fs = FactScorer(model_name=st,
            data_dir=data_dir,
            vllm_model=vllm_model,
            atomic_facts_cache_dir=atomic_facts_cache_dir,
            dump_every_int=20,
            cache_dir=cache_dir,
            abstain_detection_type=abstain_type,
            is_bio=is_bio,
            retrieval_device="cuda:0",
            context_type=context_type,
            context_num_pages=cnp,
            num_supporting_contexts=nsp,
            debug=True)

    topics, generations = load_simple_json(json_p)

    topics = topics[:5]
    generations = generations[:5]
    all_decisions = []

    fs.register_knowledge_source('enwiki-20230401', db_path='./wikidump/enwiki-20230401.db', data_path=f'data/{gn}.jsonl')
    out = fs.get_score(topics, generations, gamma=10, knowledge_source="enwiki-20230401", verbose=True)

    print(f'Score: {out["score"]}\nRespond ratio: {out["respond_ratio"]}')

Evaluating ChatGPT
atomic_facts_cache_dir='./cache/ChatGPT-retrieval+llama-wikipedia_api-p5-c10/'


 83%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                                       | 5/6 [00:14<00:02,  2.45s/it]/disk/4tb/sushko/conda/envs/ltf/lib/python3.11/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("html.parser"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /disk/4tb/sushko/conda/envs/ltf/lib/python3.11/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="html.parser"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')
100%|███████████████████████

Score: 0.317545412813955
Respond ratio: 0.6666666666666666
Evaluating InstructGPT
atomic_facts_cache_dir='./cache/InstructGPT-retrieval+llama-wikipedia_api-p5-c10/'


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:24<00:00,  4.86s/it]

Score: 0.18333634598003962
Respond ratio: 1.0
